# 07. 같은 지점 비교 분석 (case-crossover)

입력: `data/3_결과/train_table.csv` (05 노트북 결과)

**무엇을 하나**
- 사고 1건마다 **같은 자리**의 "평소" 3~4개(같은 달·같은 요일·같은 시각의 다른 주)를 묶음으로 만듦
- 묶음 안에서만 비교 → 도로 모양·터널 자체 같은 **장소 차이는 지워짐**
- 남는 것: 그날그날 달라지는 것(햇빛 → A, 비)의 효과

**결과 읽는 법**
- `A 0.1당 1.38배` = A가 0.1 오른 날은 같은 자리 평소보다 사고 오즈가 1.38배
- 95% 범위가 1을 포함하면 "아직 확실하지 않음"

**비(rain)는 왜 넣나** (목표가 아님)
1. 저울 점검: "비 → 사고↑"가 나오면 방법이 작동한다는 뜻
2. 비 오는 날은 어두워 A가 낮음 → 비를 따로 빼야 비 효과가 A에 섞이지 않음

> 초안 · 업데이트 중 (검토 전). 교통량(VDS)·길이 그룹이 들어가면 숫자는 바뀔 수 있음

In [ ]:
import pandas as pd, numpy as np, warnings
from pathlib import Path
from statsmodels.discrete.conditional_models import ConditionalLogit
warnings.filterwarnings('ignore')

DATA = Path('data')
t = pd.read_csv(DATA / '3_결과' / 'train_table.csv', low_memory=False)

# 사고(1) + 같은 지점 평소(0)만, 평소가 1개 이상 있는 사고만
cc = t[t.design.isin(['사고', '같은지점'])].copy()
cc = cc[cc.case_id.isin(cc[cc.y == 0].case_id)]
cc['rain'] = (cc.rn.fillna(0) > 0).astype(int)
cc['vs_km'] = cc.vs / 100
cc['hol'] = cc.is_holiday.astype(int)
print('사고', cc[cc.y == 1].case_id.nunique(), '건 / 평소', (cc.y == 0).sum(), '행')

In [ ]:
def fit(d, cols, label):
    """조건부 로지스틱: 사고와 그 사고의 평소끼리만 비교. 결과는 배수(오즈비)."""
    d = d.dropna(subset=cols)
    d = d[d.groupby('case_id').y.transform('sum') == 1]
    d = d[d.groupby('case_id').y.transform('size') > 1]
    m = ConditionalLogit(d.y, d[cols], groups=d.case_id).fit(disp=0)
    ci = m.conf_int()
    rows = []
    for c in cols:
        rows.append({'항목': c, '배수': round(np.exp(m.params[c]), 2),
                     '95%_하한': round(np.exp(ci.loc[c, 0]), 2),
                     '95%_상한': round(np.exp(ci.loc[c, 1]), 2),
                     'p': round(m.pvalues[c], 3)})
    print(f'== {label}: 사고 {d.case_id.nunique()}건')
    return pd.DataFrame(rows)

## 1. 저울 점검: 전체 사고에서 비·시정·공휴일

In [ ]:
fit(cc, ['rain', 'vs_km', 'hol'], '전체 (방법 점검)')

## 2. 구간별 사고 수 (낮 = 태양 고도 > 0)

In [ ]:
order = ['진입전150', '경계100', '이행300', '기본부', '출구60', '진출후150', '일반']
case = cc[cc.y == 1]
pd.crosstab(case.tunnel_zone, np.where(case.sun_elev > 0, '낮', '밤')).reindex(order)

## 3. 핵심: A가 오르면 사고 위험이 몇 배?

- 진입 A: 진입전150 + 경계100 구간 낮 사고
- 진출 A: 출구60 + 진출후150 구간 낮 사고
- A를 10배 해서 넣음 → 결과가 "A 0.1당 배수"

In [ ]:
res = {}
for zones, col, lab in [(['진입전150', '경계100'], 'A_진입', '진입'),
                        (['출구60', '진출후150'], 'A_진출', '진출')]:
    z = cc[cc.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].copy()
    d['A10'] = d[col] * 10
    res[lab] = fit(d, ['A10', 'rain'], f'{lab} 낮, A 0.1당')
    display(res[lab])

## 4. 발표용: A를 3등급으로 나눠 비교 (진입)

| 등급 | 밝기 차이(바깥÷터널 안) | A |
|---|---|---|
| 낮음 | 10배 미만 | 0.50 미만 |
| 보통 | 10~15배 | 0.50~0.59 |
| 높음 | 15배 이상 | 0.59 이상 |

기준 = 낮음. "보통·높음 날은 낮음 날보다 몇 배"

In [ ]:
z = cc[cc.tunnel_zone.isin(['진입전150', '경계100'])]
day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
d = z[z.case_id.isin(day_cases)].copy()
d['등급'] = pd.cut(d['A_진입'], [-1, 0.5, np.log10(15) / 2, 2], right=False,
                 labels=['낮음', '보통', '높음'])
print(pd.crosstab(d['등급'], d.y.map({1: '사고', 0: '평소'})))
x = pd.get_dummies(d, columns=['등급'], drop_first=True, dtype=int)
fit(x, ['등급_보통', '등급_높음', 'rain'], '진입 낮, 등급별 (기준 낮음)')

## 5. 해석 메모

- 진입: 밝을수록 위험한 방향. 범위가 1을 살짝 포함 → "경향" 수준
- 진출: 아직 신호 없음 (낮 사고 64건으로 적음)
- 다음: 교통량(VDS) 추가, 터널 길이 그룹별 비교, 같은 날 ±1~2시간 보조 분석